read dde from kn.her.all, first column includes two single mutations aa and site information, second column is dde, third is DE_double, fourth is first de, fifth is second de

# 1. read dde from file


In [85]:
import pandas as pd
# Try reading the file with space as a delimiter
df = pd.read_csv('../data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])
# Split the 'Mutation' column into 'First_mutation' and 'Second_mutation'
df[['First_mutation', 'Second_mutation']] = df['Mutation'].str.split('-', expand=True)
# Display the DataFrame
print(df)


/var/folders/17/rj19bvws2qscyfjmb7m44zmm0000gn/T/ipykernel_81417/1655497556.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv('../data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])


           Mutation       DDE  DE_double  First_DE  Second_DE First_mutation  \
0           A1B-B2A  0.012168 -20.532304 -8.380550 -12.211337            A1B   
1           A1B-B2C -0.000908 -16.498453 -8.380550  -8.179655            A1B   
2           A1B-B2D  0.000125 -16.658839 -8.380550  -8.339867            A1B   
3           A1B-A3B  0.008785 -14.738873 -8.380550  -6.419735            A1B   
4           A1B-A3C  0.008937 -13.811957 -8.380550  -5.493224            A1B   
...             ...       ...        ...       ...        ...            ...   
310072  B262C-A263C  0.390912 -16.406621 -8.433813  -8.236725          B262C   
310073  B262C-A263D  0.391073 -16.553047 -8.433813  -8.383161          B262C   
310074  B262D-A263B  0.267580 -15.236453 -8.310417  -7.223255          B262D   
310075  B262D-A263C  0.001986 -16.321536 -8.310417  -8.236725          B262D   
310076  B262D-A263D  0.002057 -16.467874 -8.310417  -8.383161          B262D   

       Second_mutation  
0             

In [86]:
# Filter rows for D148B and C140D
filtered_dde = df[(df['First_mutation'] == 'D148B') & (df['Second_mutation'] == 'C140D') |
                  (df['First_mutation'] == 'C140D') & (df['Second_mutation'] == 'D148B')]

# Print the DDE values
print(filtered_dde)

           Mutation      DDE  DE_double  First_DE  Second_DE First_mutation  \
242203  C140D-D148B  8.50879 -11.219437 -6.705909  -4.577816          C140D   

       Second_mutation  
242203           D148B  


# 2. index each sequence by its mutations compared to in.consensus.reduce4.seq

read in from ../data/in.reduce4.seq, there are 1220 sequences, create a df, with last column as mutations: with a list of mutaiton that occored compared to ../data/in.consensus.reduce4.seq the mutations are in format for example: [D148B, C140D, ...] where D and C notes the wildtime from consensus at pos 148 and 140

In [87]:
# Read the consensus sequence
with open('../data/in.consensus.reduce4.seq', 'r') as f:
    consensus_sequence = f.read().strip()

# Read the 1220 sequences
sequences = []
with open('../data/in.reduce4.seq', 'r') as f:
    for line in f:
        sequences.append(line.strip())

# Create a DataFrame to store sequences and their mutations
sequence_df = pd.DataFrame({'Sequence': sequences})

# Function to identify mutations compared to the consensus sequence
def find_mutations(sequence, consensus):
    mutations = []
    for i, (seq_residue, cons_residue) in enumerate(zip(sequence, consensus), start=1):
        if seq_residue != cons_residue:
            mutations.append(f"{cons_residue}{i}{seq_residue}")
    return mutations

# Add a column for mutations
sequence_df['Mutations'] = sequence_df['Sequence'].apply(lambda seq: find_mutations(seq, consensus_sequence))
sequence_df['Mutations_count'] = sequence_df['Mutations'].apply(len)
# Display the DataFrame
print(sequence_df)

                                               Sequence  \
0     ABAAABABACBBAACBDBABBDBDBBBAAAAAABAAABDAABAABA...   
1     ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...   
2     ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...   
3     ABAAABCBACABAACBABABBDBDBBBBAACAABAAABAAABAABA...   
4     ABAAABCBABABAACBDBABBDBDBBBAAACAABAAABDAABAABA...   
...                                                 ...   
1215  ABAAABCBACABAACBABABBDBDBBBBAACAABAAABDAABAAAA...   
1216  ABAAABCBACABAACBDBABBDBDBBBAAACAABAAABDAABAAAA...   
1217  ABAAABCBACABAACBABABBDBDBBBAAAAAABAAABDAABAABA...   
1218  ABAAABCBACABAACBDBABBDBDBBBAAAAAABAAABDAABAABA...   
1219  ABAAABCBACBBAACBDBABBDBDBBBBAAABABAABBDAABAABA...   

                                              Mutations  Mutations_count  
0     [C7A, A11B, C31A, C50B, A72D, A101B, C124A, B1...               13  
1     [A11B, B21C, B25A, D119A, C122B, D125A, D148C,...               13  
2     [A11B, B21C, B25A, D119A, C122B, D125A, C140D,...           

# 3. J matrix and delta e definition

## 3.1 J matrix

In [88]:
import numpy as np

#dictionary of J matrix
J_dict = {}

# Load the J matrix from the downloaded file
J = np.load('../data/J.npy')

row = 0
for pos1 in range(1, 264):
    for pos2 in range(pos1 + 1, 264):

        for i, aa1 in enumerate(['A', 'B', 'C', 'D']):
            for j, aa2 in enumerate(['A', 'B', 'C', 'D']):
                col = i*4+j


                J_dict[(pos1, pos2, aa1, aa2)] = J[row, col]
                J_dict[(pos2, pos1, aa2, aa1)] = J[row, col]
            # print(f"Row {row}: J[{row}, {col}] = {J[row, col]} for positions ({pos1}, {pos2}) with amino acids ({aa1}, {aa2})")
        row += 1

print(f"Dictionary created with {len(J_dict)} entries")

Dictionary created with 1102496 entries


## 3.2 define delta e

In [89]:
# Define delta E calculation
def calculate_delta_e(position, old_amino_acid, new_amino_acid, seq, J_dict):
    # E(old_amino_acid)
    energy_old = 0
    for other_pos in range(1, 264):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - 1]  # Access the first sequence in sequence_list
        energy_old += J_dict.get((position, other_pos, old_amino_acid, other_aa), 0)

    # E(new_amino_acid)
    energy_new = 0
    for other_pos in range(1, 264):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - 1]  # Access the first sequence in sequence_list
        energy_new += J_dict.get((position, other_pos, new_amino_acid, other_aa), 0)

    delta_e = energy_old - energy_new
    # print(f"E({old_amino_acid}) at {position}: {energy_old}")
    # print(f"E({new_amino_acid}) at {position}: {energy_new}")
    # print(f"Delta E for {old_amino_acid}{position}{new_amino_acid}: {delta_e}")
    return delta_e

# # Example usage
# position = 140
# old_amino_acid = 'C'
# new_amino_acid = 'D'
# calculate_delta_e(position, old_amino_acid, new_amino_acid, sequence_list, J_dict)

In [90]:
# define dm12
def calculate_dm12 (pos1, old_amino_acid1, new_amino_acid1, pos2, old_amino_acid2, new_amino_acid2, seq, J_dict):
    energy_old = 0
    energy_new = 0

# old energy
    for other_pos in range(1, 264):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_old += J_dict.get((pos1, pos2, old_amino_acid1, old_amino_acid2), 0)
        else:
            energy_old += J_dict.get((pos1, other_pos, old_amino_acid1, other_aa), 0)

    for other_pos in range(1, 264):
        other_aa = seq[other_pos - 1]
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            # energy_old += J_dict.get((pos2, pos1, old_amino_acid2, old_amino_acid1), 0)
            continue
        else:
            energy_old += J_dict.get((pos2, other_pos, old_amino_acid2, other_aa), 0)

# new energy
    for other_pos in range(1, 264):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_new += J_dict.get((pos1, pos2, new_amino_acid1, new_amino_acid2), 0)
        else:
            energy_new += J_dict.get((pos1, other_pos, new_amino_acid1, other_aa), 0)

    for other_pos in range(1, 264):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            # energy_new += J_dict.get((pos2, pos1, new_amino_acid2, new_amino_acid1), 0)
            continue
        
        else:
            energy_new += J_dict.get((pos2, other_pos, new_amino_acid2, other_aa), 0)
    
    return energy_old - energy_new




## 3.3 run de ono all 1220

run de on all 1220 sequences with D148B, C140D (C140D, D148B), if the consensus vs one of 1220 sequences where the DE of each mutation changes its size compare to others  ( DE -DE sign change from consensus where the sign change means the sign of de D148B- deC140D sign change), record that sequence with its mutaitons and mutation len

In [95]:
# List to store sequences with sign change
sequences_with_sign_change = []
# consensus_sequence = 'ABAAABABACBBAACBDBABBDBDBBBAAAAAABAAABDAABAABAAABBBAAAAACACBAABAADACCBADAACACABBABABAAABABBAABDAABCABAABABAAAABDCBCAAADABCCADDAAAAAAAACDBDADBADAAABBABAAAAACBBADAAACCAAABAAABAAABAABABAABABABAAAACCAAAAAAAABBBBCACDCAACDBDCBCAAAAABADDBCACAAAAABAABAAAAAAABBDBDBACCBBBA'
consensus_sequence = 'ABAAABCBACABAACBDBABBDBDBBBAAACAABAAABDAABAABAAABCBAAAAACACBAABAADACCBAAAACACABBABABAAABABBAABDAABCAAAABABAAAABDCBCAAADABCCCDDAAAAAAAABDBDACBADAAABDABAAAAACBBADAAACCAAABAAABAAABAABABAABABABAAAACCAAAABBAABBBBCACDCAACDBDCCCAAAAABADDBCACAAAAABAABAAAAAAABBDBDBACCBBBA'
de_d148b_consensus = calculate_delta_e(148, 'D', 'B', consensus_sequence, J_dict)
de_c140d_consensus = calculate_delta_e(140, 'C', 'D', consensus_sequence, J_dict)
print(de_c140d_consensus,de_d148b_consensus)
print(calculate_dm12(140, 'C', 'D', 148, 'D', 'B', consensus_sequence, J_dict))
# Iterate through all sequences
for index, row in sequence_df.iterrows():
    sequence = row['Sequence']
    mutations = row['Mutations']
    mutation_count = row['Mutations_count']
    
    # Skip sequences without the specified mutations
    if 'D148B' not in mutations or 'C140D' not in mutations:
        continue
        # pass
    
    # Calculate delta E for D148B and C140D
    de_d148b = calculate_delta_e(148, 'D', 'B', sequence, J_dict)
    de_c140d = calculate_delta_e(140, 'C', 'D', sequence, J_dict)

    # print(de_d148b)
    # print(de_c140d)
    
    # Calculate delta E for consensus
    # de_d148b_consensus = calculate_delta_e(148, 'D', 'B', consensus_sequence, J_dict)
    # de_c140d_consensus = calculate_delta_e(140, 'C', 'D', consensus_sequence, J_dict)
    
    # Check for sign change
    if (de_d148b - de_c140d) * (de_d148b_consensus - de_c140d_consensus) < 0:
        sequences_with_sign_change.append({
            'Sequence': sequence,
            'Mutations': mutations,
            'Mutation_count': mutation_count
        })

# Create a DataFrame to store the results
sign_change_df = pd.DataFrame(sequences_with_sign_change)

# Display the resulting DataFrame
# Extract and print the mutations along with their DE values
for _, row in sign_change_df.iterrows():
    mutations = row['Mutations']
    sequence = row['Sequence']
    print("Mutations:", mutations)
    for mutation in mutations:
        if mutation != 'C140D' and mutation != 'D148B':
            continue
        pos = int(mutation[1:-1])  # Extract position from mutation string
        old_aa = mutation[0]      # Extract old amino acid
        new_aa = mutation[-1]     # Extract new amino acid
        de1 = calculate_delta_e(pos, old_aa, new_aa, sequence, J_dict)
        print(f"  {mutation}: DE1 = {de1}")
    de12 = calculate_dm12(140, 'C', 'D', 148, 'D', 'B', sequence, J_dict)
    print(f"  DE12 (C140D, D148B): {de12}")

-5.7286787 -4.553505
-1.7733994
Mutations: ['B63A', 'A73B', 'A74C', 'A96B', 'A101B', 'D112B', 'C140D', 'D148B', 'C208A', 'C232A']
  C140D: DE1 = 5.339542865753174
  D148B: DE1 = 4.223781108856201
  DE12 (C140D, D148B): 1.0545291900634766
Mutations: ['B63A', 'A73B', 'A74C', 'A96B', 'A101B', 'D112B', 'C140D', 'D148B', 'C208A', 'C232A']
  C140D: DE1 = 5.339542865753174
  D148B: DE1 = 4.223781108856201
  DE12 (C140D, D148B): 1.0545291900634766
Mutations: ['A11B', 'D24C', 'C31A', 'D112B', 'C140D', 'D148B', 'B206A', 'C208A']
  C140D: DE1 = 4.067683696746826
  D148B: DE1 = 3.63800048828125
  DE12 (C140D, D148B): -0.8031101226806641
Mutations: ['D24C', 'A41C', 'A74C', 'B91C', 'A101B', 'D125A', 'C140D', 'D148B', 'B201A', 'D216A', 'B254D']
  C140D: DE1 = 5.296130180358887
  D148B: DE1 = 5.244509220123291
  DE12 (C140D, D148B): 2.031848907470703
Mutations: ['D17A', 'A32B', 'A74C', 'C124A', 'D125B', 'D136B', 'C140D', 'D148B', 'C165A', 'B201A', 'C234A', 'B256A', 'C259A']
  C140D: DE1 = 5.5584640502